# Pydantic -- Practical Guide

**Pydantic** = Python library for **data validation using type hints**.

Simple ga cheppalante:
> Oka class define chesthe -- Pydantic automatically wrong data ki error ivvunu, correct data ki clean object ivvunu.

---

## Why Pydantic?

Normal Python class lo wrong data ichhinappudu **no error** -- silently accept avutundi.
Pydantic `BaseModel` tho wrong data ichhinappudu **immediate clear ValidationError** vastundi.

**Install:**
```
pip install pydantic
```

## 1) BaseModel -- Core Concept

`BaseModel` ante Pydantic lo anni data models ki **parent class**.

- Oka class create chestam `BaseModel` inherit chesi
- Fields define chestam **type hints** tho (`name: str`, `age: int`)
- Pydantic automatic ga validate chesthundi -- wrong type = **ValidationError**

**Analogy:**
> Bank form laanti di. Form lo `Name` field lo number rayyataniki allow cheyyadam ledu -- same way `name: str` ki int value pass cheyyataniki allow cheyyadam ledu.

In [22]:
from pydantic import BaseModel

# Person model define chestunnam
# name -> must be string
# age  -> must be integer
# city -> must be string
class Person(BaseModel):
    name: str
    age: int
    city: str

# Correct data tho object create cheyyadam
person = Person(name='Krish', age=35, city='Bangalore')

# print cheste Pydantic clean formatted output istundi
print(person)

name='Krish' age=35 city='Bangalore'


### Output Explanation

```
name='Krish' age=35 city='Bangalore'
```

Pydantic `print(person)` chessinappudu **field=value** format lo clean output istundi.
Normal class laaga `<__main__.Person object at 0x...>` kaadu -- human-readable format.

## 2) Accessing Fields

Object create ayyaka fields ni normal attribute access tho use cheyyachu.

- `person.name` -> field value
- `person.age`  -> guaranteed int (type safe)
- `person.city` -> guaranteed str

**Key point:** Pydantic validate chesindi kabatti -- `person.age` is always `int`, kabatti `person.age + 1` always works safely.

In [23]:
# Individual fields access cheyyadam
print('Name :', person.name)
print('Age  :', person.age)
print('City :', person.city)

# Type check -- guaranteed correct types
print(type(person.name))  # -> <class 'str'>
print(type(person.age))   # -> <class 'int'>

# Safe arithmetic -- age is int, kabatti + works
print('Age next year:', person.age + 1)

Name : Krish
Age  : 35
City : Bangalore
<class 'str'>
<class 'int'>
Age next year: 36


## 3) Auto Type Conversion

Pydantic **intelligent** -- reasonable conversions automatic ga chesthundi.

| Input | Field Type | Result |
|-------|-----------|--------|
| `'35'` (string) | `int` | `35` (int) auto convert |
| `35` (int) | `str` | `'35'` (str) auto convert |
| `'abc'` (string) | `int` | ValidationError |

String ga number ichhinappudu -- int ga convert chesthundi. Impossible conversions ki error istundi.

In [24]:
# Auto type conversion demo
# age ki string '28' ichham -- Pydantic int 28 ga convert chesthundi
person2 = Person(name='Rahul', age='28', city='Mumbai')
print(person2.age)         # -> 28
print(type(person2.age))   # -> <class 'int'>  (string '28' -> int 28 automatic!)

28
<class 'int'>


## 4) ValidationError -- Wrong Data

Impossible conversion try chessinappudu Pydantic **ValidationError** raise chesthundi.

- Which field wrong -- clearly chepthundi
- What was expected -- chepthundi
- What was received -- chepthundi

Silent failure ledu -- **fail fast, fail clearly**.

In [25]:
from pydantic import ValidationError

# Wrong data -- age ki 'thirty-five' ichham
# int ga convert impossible -- ValidationError raise avutundi
try:
    bad_person = Person(name='Test', age='thirty-five', city='Delhi')
except ValidationError as e:
    print('Validation failed!')
    print(e)

Validation failed!
1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='thirty-five', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


## 5) dataclass vs Pydantic BaseModel -- Key Difference

Python built-in `@dataclass` kuda similar ga data classes create cheyyadam allow chesthundi.
Kaani Pydantic `BaseModel` tho **chaala important difference** undi.

| Feature | `@dataclass` | Pydantic `BaseModel` |
|---------|-------------|----------------------|
| Syntax | Similar | Similar |
| Auto `__init__` | Yes | Yes |
| Auto `__repr__` | Yes | Yes |
| **Type validation** | **No** | **Yes** |
| **Wrong type = error** | **No -- silent** | **Yes -- ValidationError** |
| Auto type conversion | No | Yes |
| `model_dump()` / JSON | No (manual) | Built-in |

**Output format difference:**
- `@dataclass` print: `Person(name='Krish', age=35, city='Bangalore')`  <- class name prefix
- `BaseModel` print:  `name='Krish' age=35 city='Bangalore'`

Rendu similar ga chustay -- kaani `@dataclass` wrong data ki **error ivvadu**. Pydantic ivvunu.

In [26]:
from dataclasses import dataclass

# @dataclass decorator -- Python built-in way
@dataclass
class PersonDC():
    name: str
    age: int
    city: str

person_dc = PersonDC(name="Krish", age=35, city="Bangalore")
print(person_dc)  # Person(name='Krish', age=35, city='Bangalore')

# PROBLEM: @dataclass lo wrong type ichhinappudu -- NO error!
person_dc2 = PersonDC(name="Krish", age=35, city=12)  # city should be str but int ichham
print(person_dc2)  # silently accepts!  <-- DANGEROUS
print(type(person_dc2.city))  # -> <class 'int'>  (str expected kaani int store chesindi!)

PersonDC(name='Krish', age=35, city='Bangalore')
PersonDC(name='Krish', age=35, city=12)
<class 'int'>


### dataclass Output Explanation

```
PersonDC(name='Krish', age=35, city='Bangalore')
PersonDC(name='Krish', age=35, city=12)   <- No error! city=12 silently stored
<class 'int'>                              <- city is int, not str!
```

`@dataclass` type hints ni **just documentation** ga treat chesthundi -- enforce cheyyadu.
`city: str` raasam -- kaani `12` (int) pass chessinappudu **no error, no conversion**.

Idi production code lo **silent bug** avutundi -- later code `city.upper()` call chesthe crash avutundi.

## 6) Pydantic Catches Wrong Type -- city=12

Same scenario -- `city=12` (int) pass chesthe Pydantic `BaseModel` ela react chesthundo chuddam.

`@dataclass`: silently accept -> silent bug
Pydantic `BaseModel`: **immediate ValidationError** -> bug early catch

Ee concept image lo chupinchindi -- `city=12` pass chessinappudu `[11]` cell lo red cross marker tho ValidationError vastundi.

In [27]:
from pydantic import BaseModel, ValidationError

class Person(BaseModel):
    name: str
    age: int
    city: str

# city ki int 12 pass chestunnam -- str expected
try:
    person1 = Person(name="Krish", age=35, city=12)
    print(person1)  # ee line reach avvadu
except ValidationError as e:
    print("Pydantic caught the error!")
    print(e)

Pydantic caught the error!
1 validation error for Person
city
  Input should be a valid string [type=string_type, input_value=12, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


### Output Explanation

```
Pydantic caught the error!
1 validation error for Person
city
  Input should be a valid string [type=string_type, ...]
```

**Wait -- city=12, int ki str ga convert cheyyachu kadha?**

Pydantic v2 lo **strict behavior for str fields** -- `int` ni automatically `str` ga convert cheyyadu (unlike `age` field where `'28'` -> `28` worked).

Reason:
- `int` -> `str` conversion: ambiguous -- `12` -> `'12'`? App logic depend chestuundi
- `str` -> `int` conversion: safe only if digits -- `'28'` -> `28` clear

Idi actually **safer behavior** -- unexpected int ni str ga silently convert chesthe logic bugs vastay.

### dataclass vs Pydantic Final Comparison:

```python
# dataclass -- silent fail
person_dc = PersonDC(name='Krish', age=35, city=12)  # No error! city=12 stored

# Pydantic -- loud fail
person_p  = Person(name='Krish', age=35, city=12)    # ValidationError immediately!
```

**Pydantic = Fail Fast** -- wrong data enter chessinappude catch chestundi, deep lo crash avvadu.

## 5) model_dump() -- Object to Dict

`model_dump()` -> Pydantic object ni Python **dictionary** ga convert chestundi.

Use cases:
- API response ga return cheyyadam
- JSON ga convert cheyyadam
- Database lo save cheyyadam
- Another function ki pass cheyyadam

In [28]:
# model_dump() -- object to dict
person_dict = person.model_dump()
print(person_dict)
print(type(person_dict))  # -> <class 'dict'>

# Dict fields access
print(person_dict['name'])  # -> 'Krish'
print(person_dict['age'])   # -> 35

{'name': 'Krish', 'age': 35, 'city': 'Bangalore'}
<class 'dict'>
Krish
35


## 6) model_dump_json() and model_validate()

`model_dump_json()` -> Pydantic object ni **JSON string** ga convert chestundi.

Difference:
- `model_dump()` -> Python `dict` (in-memory use)
- `model_dump_json()` -> JSON `str` (network send, file save)

`model_validate()` -> reverse -- **dict to Pydantic object**.

In [29]:
# model_dump_json() -- object to JSON string
person_json = person.model_dump_json()
print(person_json)
print(type(person_json))  # -> <class 'str'>

# model_validate() -- dict to object (reverse of model_dump)
data = {'name': 'Anita', 'age': 30, 'city': 'Chennai'}
person3 = Person.model_validate(data)
print(person3)  # -> name='Anita' age=30 city='Chennai'

{"name":"Krish","age":35,"city":"Bangalore"}
<class 'str'>
name='Anita' age=30 city='Chennai'


## 7) Optional Fields -- Fields That May or May Not Have a Value

Real world lo anni fields always required undavu.
Example: Employee lo `salary` inka decide cheyyaledu, `is_active` default True.

`Optional[type]` use chesthe -- aa field **required kaadu**, `None` pass cheyyachu or skip cheyyachu.

```python
from typing import Optional

salary: Optional[float] = None   # not required, default None
is_active: Optional[bool] = True # not required, default True
```

| Field | Type | Default | Required? |
|-------|------|---------|----------|
| `id` | `int` | -- | Yes |
| `name` | `str` | -- | Yes |
| `department` | `str` | -- | Yes |
| `salary` | `Optional[float]` | `None` | No |
| `is_active` | `Optional[bool]` | `True` | No |

**Key rule:** Default value unna fields ni skip cheyyachu -- Pydantic default use chesthundi.

In [30]:
from pydantic import BaseModel
from typing import Optional

class Employee(BaseModel):
    id: int
    name: str
    department: str
    salary: Optional[float] = None   # Optional with default None
    is_active: Optional[bool] = True  # Optional with default True

# emp1 -- only required fields, optional ones skip chestunnam
emp1 = Employee(id=1, name="John", department="IT")
print(emp1)  # salary=None  is_active=True  (defaults used)

id=1 name='John' department='IT' salary=None is_active=True


### emp1 Output Explanation

```
id=1 name='John' department='IT' salary=None is_active=True
```

- `salary` pass cheyyaledu -> default `None` use chesindi
- `is_active` pass cheyyaledu -> default `True` use chesindi
- Required fields (`id`, `name`, `department`) pass chessinappude object create avutundi

In [31]:
# emp2 -- anni fields provide chestunnam
emp2 = Employee(id=2, name="Jane", department="HR", salary=60000.0, is_active=False)
print(emp2)  # salary=60000.0  is_active=False

id=2 name='Jane' department='HR' salary=60000.0 is_active=False


### emp2 Output Explanation

```
id=2 name='Jane' department='HR' salary=60000.0 is_active=False
```

Anni fields provide chessinamu -- defaults override ayyay.
- `salary=60000.0` -> provided value use chesindi
- `is_active=False` -> provided value use chesindi (default `True` override)

In [32]:
# Auto conversion demo -- salary ki int 60000 icham, float ga convert avutundi
emp3 = Employee(id=2, name="Jane", department="HR", salary=60000, is_active=False)
print(emp3)              # salary=60000.0  (int 60000 -> float 60000.0 automatic!)
print(type(emp3.salary)) # -> <class 'float'>

# Comparison
print("\n--- Summary ---")
print(f"emp1 salary: {emp1.salary}  (None -- not provided)")
print(f"emp2 salary: {emp2.salary}  (float provided)")
print(f"emp3 salary: {emp3.salary}  (int 60000 -> float 60000.0 auto converted)")

id=2 name='Jane' department='HR' salary=60000.0 is_active=False
<class 'float'>

--- Summary ---
emp1 salary: None  (None -- not provided)
emp2 salary: 60000.0  (float provided)
emp3 salary: 60000.0  (int 60000 -> float 60000.0 auto converted)


### Auto Conversion -- int to float

```
salary=60000.0         # int 60000 -> float 60000.0 automatic!
<class 'float'>
```

Image 3 lo exactly ee behavior chupinchindi -- `salary=60000` (int) pass chessinamu, output lo `salary=60000.0` (float) ga convert ayyindi.

**Why Pydantic converts int -> float?**
- `int` is always representable as `float` -- no data loss
- `60000` -> `60000.0` completely safe conversion
- Opposite sometimes not safe: `60000.7` (float) -> `60000` (int) = data loss!

**Optional field cheat sheet:**
```python
from typing import Optional

# None default -- not provided aithe None
salary: Optional[float] = None

# Custom default -- not provided aithe True
is_active: Optional[bool] = True

# Required -- must provide
name: str   # no default = required
```

## 8) List Fields -- List[str], List[int], List[Model]

Pydantic lo field type `List[str]` ga define cheyyachu -- ante aa field lo **list of strings** matrame accept avutundi.

```python
from typing import List

students: List[str]   # list of strings only
scores:   List[int]   # list of integers only
```

**Validation chesthundi:**
- `students` ki `['Alice', 'Bob']` --> OK
- `students` ki `['Alice', 123]`   --> ValidationError (123 is not str)
- `students` ki `'Alice'`          --> ValidationError (list kaadu)

**Analogy:**
> Classroom register lo student names only untay -- numbers or random data allow cheyyadu.

In [33]:
from pydantic import BaseModel
from typing import List

class Classroom(BaseModel):
    room_number: str
    students: List[str]  # List of strings
    capacity: int

# Create a classroom
classroom = Classroom(
    room_number="A101",
    students=["Alice", "Bob", "Charlie"],
    capacity=30
)
print(classroom)

room_number='A101' students=['Alice', 'Bob', 'Charlie'] capacity=30


### Output Explanation

```
room_number='A101' students=['Alice', 'Bob', 'Charlie'] capacity=30
```

- `room_number` -> str field, `'A101'` stored
- `students` -> `List[str]` field, full list stored as-is
- `capacity` -> int field, `30` stored

In [34]:
# Accessing list field -- normal Python list operations anni work avutay
print(classroom.students)          # -> ['Alice', 'Bob', 'Charlie']
print(classroom.students[0])       # -> 'Alice'
print(classroom.students[-1])      # -> 'Charlie'
print(len(classroom.students))     # -> 3

# Loop cheyyadam
for student in classroom.students:
    print(f"  Student: {student}")

# Capacity check
print(f"\nSeats filled : {len(classroom.students)}")
print(f"Total seats  : {classroom.capacity}")
print(f"Seats left   : {classroom.capacity - len(classroom.students)}")

['Alice', 'Bob', 'Charlie']
Alice
Charlie
3
  Student: Alice
  Student: Bob
  Student: Charlie

Seats filled : 3
Total seats  : 30
Seats left   : 27


### List Field -- Validation Demo

Pydantic `List[str]` field ki wrong data ichhinappudu error ivvunu.

In [35]:
from pydantic import ValidationError

# Wrong: students lo int value ichham
try:
    bad_class = Classroom(
        room_number="B202",
        students=["Alice", 123, "Charlie"],  # 123 is int, not str
        capacity=25
    )
except ValidationError as e:
    print("Validation Error!")
    print(e)

print("-" * 40)

# model_dump() -- list field also included in dict
classroom_dict = classroom.model_dump()
print(classroom_dict)
print(type(classroom_dict["students"]))  # -> <class 'list'>

Validation Error!
1 validation error for Classroom
students.1
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
----------------------------------------
{'room_number': 'A101', 'students': ['Alice', 'Bob', 'Charlie'], 'capacity': 30}
<class 'list'>


### Key Takeaways -- List Fields

| Scenario | Result |
|----------|--------|
| `students=['Alice', 'Bob']` | OK -- all strings |
| `students=['Alice', 123]` | ValidationError -- 123 not str |
| `students='Alice'` | ValidationError -- string, not a list |
| `students=[]` | OK -- empty list allowed |

```python
# Common List types in Pydantic
students:  List[str]       # list of strings
scores:    List[int]       # list of integers
ratings:   List[float]     # list of floats
tags:      List[str]       # tags, labels etc.
ids:       List[int]       # list of IDs
```

**Bottom line:** `List[str]` means every item in the list must be a `str` -- Pydantic each item validate chesthundi.

## 9) Nested Models -- Model Inside Model

Real world data often **hierarchical** ga untundi.
Example: Customer ki Address untundi, Order ki Customer + Items untay.

Pydantic lo oka model ni inkoka model lo field ga use cheyyachu -- **Nested Models**.

```python
class Address(BaseModel):     # inner model
    street: str
    city: str
    zip_code: str

class Customer(BaseModel):    # outer model
    customer_id: int
    name: str
    address: Address          # Address model as a field
```

**Key feature:** `address` field ki **dict ga** pass cheyyachu -- Pydantic automatically `Address` object ga convert chesthundi.

**Analogy:**
> Courier form lo sender details oka box lo untay, receiver details inko box lo. Box inside box -- same way model inside model.

In [36]:
from pydantic import BaseModel

class Address(BaseModel):
    street: str
    city: str
    zip_code: str

class Customer(BaseModel):
    customer_id: int
    name: str
    address: Address  # Nested model

# Create a customer with nested address
# address ki dict pass chestunnam -- Pydantic Address object ga convert chesthundi
customer = Customer(
    customer_id=1,
    name="Emma",
    address={"street": "123 Main St", "city": "Boston", "zip_code": "02108"}
)
print(customer)

customer_id=1 name='Emma' address=Address(street='123 Main St', city='Boston', zip_code='02108')


### Output Explanation

```
customer_id=1 name='Emma' address=Address(street='123 Main St' city='Boston' zip_code='02108')
```

- `address` field lo `Address` object stored -- dict ga pass chessinamu, Pydantic convert chesindi
- Nested object print chessinappudu `Address(...)` format lo kanipistundi
- Full validation both levels lo jarugtundi -- outer + inner

In [37]:
# Nested fields access -- dot notation chain cheyyadam
print(customer.name)               # -> 'Emma'
print(customer.address)            # -> Address object
print(customer.address.city)       # -> 'Boston'
print(customer.address.street)     # -> '123 Main St'
print(customer.address.zip_code)   # -> '02108'

print(type(customer.address))      # -> <class '__main__.Address'>  (dict kaadu, Address object!)

Emma
street='123 Main St' city='Boston' zip_code='02108'
Boston
123 Main St
02108
<class '__main__.Address'>


### Dict ga pass chessinamu -- kaani Address object ga store ayyindi

Idi Pydantic nested model magic:

```python
# Pass chessinappudu -- dict
address={"street": "123 Main St", "city": "Boston", "zip_code": "02108"}

# Store ayyinappudu -- Address object
type(customer.address)  # -> <class 'Address'>
```

Faida enti ante:
- `customer.address["city"]`  -- dict access, error prone
- `customer.address.city`     -- attribute access, clean and safe ?

Pydantic dict ni object ga convert chesindi kabatti -- **dot notation** use cheyyachu, typo chessinappudu immediate error vastundi.

In [38]:
# model_dump() -- nested model kuda dict ga unfold avutundi
customer_dict = customer.model_dump()
print(customer_dict)
# -> {'customer_id': 1, 'name': 'Emma', 'address': {'street': '123 Main St', 'city': 'Boston', 'zip_code': '02108'}}

print()
# Nested dict access
print(customer_dict["address"]["city"])   # -> 'Boston'

# model_dump_json() -- full nested JSON string
print()
print(customer.model_dump_json())

{'customer_id': 1, 'name': 'Emma', 'address': {'street': '123 Main St', 'city': 'Boston', 'zip_code': '02108'}}

Boston

{"customer_id":1,"name":"Emma","address":{"street":"123 Main St","city":"Boston","zip_code":"02108"}}


In [39]:
from pydantic import ValidationError

# Nested validation -- inner model lo wrong data ichhinappudu
try:
    bad_customer = Customer(
        customer_id=2,
        name="John",
        address={"street": "456 Oak Ave", "city": "NY", "zip_code": 99999}  # zip_code int icham
    )
except ValidationError as e:
    print("Nested ValidationError!")
    print(e)
# Pydantic error lo 'address.zip_code' ga exactly eppati nested field wrong undo chepthundi

Nested ValidationError!
1 validation error for Customer
address.zip_code
  Input should be a valid string [type=string_type, input_value=99999, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


### Nested Validation Error Output

```
Nested ValidationError!
1 validation error for Customer
address.zip_code
  Input should be a valid string [type=string_type ...]
```

`address.zip_code` -- dot notation tho exactly which nested field wrong undo Pydantic chepthundi.

### Summary -- Nested Models

| Concept | Example |
|---------|--------|
| Inner model define | `class Address(BaseModel): ...` |
| Outer model use cheyyadam | `address: Address` |
| Dict ga pass cheyyachu | `address={"city": "Boston", ...}` |
| Object ga access | `customer.address.city` |
| `model_dump()` | Nested dict ga unfold avutundi |
| Validation | Both outer + inner fields validate avutay |

**Bottom line:** Nested models tho complex hierarchical data clean ga represent cheyyachu -- Pydantic dict-to-object conversion automatic, validation both levels lo jarugtundi.

## 10) Field() -- Constraints & Validation Rules

Basic type hints (`name: str`) just type check chestay.
`Field()` use chesthe **extra rules** add cheyyadam possible -- min/max length, numeric ranges, patterns.

```python
from pydantic import BaseModel, Field

name: str   = Field(min_length=2, max_length=50)  # string length rules
price: float = Field(gt=0, le=1000)               # numeric range rules
quantity: int = Field(ge=0)                       # non-negative rule
```

### Constraint Cheat Sheet:

| Constraint | Full form | Meaning | Example |
|------------|-----------|---------|--------|
| `min_length` | minimum length | String minimum characters | `Field(min_length=2)` |
| `max_length` | maximum length | String maximum characters | `Field(max_length=50)` |
| `gt` | greater than | Number > value (not equal) | `Field(gt=0)` means > 0 |
| `ge` | greater than or equal | Number >= value | `Field(ge=0)` means >= 0 |
| `lt` | less than | Number < value (not equal) | `Field(lt=100)` |
| `le` | less than or equal | Number <= value | `Field(le=1000)` means <= 1000 |
| `default` | default value | Value when not provided | `Field(default='active')` |
| `pattern` | regex pattern | String must match regex | `Field(pattern=r'^\d{10}$')` |

**Analogy:**
> Online shopping form lo -- product name minimum 2 characters undali, price 0 kante ekkuva undali, quantity negative kaadali. Avi anni `Field()` rules tho enforce cheyyadam.

In [ ]:
from pydantic import BaseModel, Field

class Item(BaseModel):
    name: str   = Field(min_length=2, max_length=50)
    price: float = Field(gt=0, le=1000)   # greater than 0, less than or equal to 1000
    quantity: int = Field(ge=0)           # greater than or equal to 0

# Valid instance
item = Item(name="Book", price=29.99, quantity=10)
print(item)

### Output

```
name='Book' price=29.99 quantity=10
```

Anni constraints satisfy chesindhi kabatti object create ayyindi:
- `'Book'` -> length 4, min_length=2 satisfy ?
- `29.99` -> gt=0 and le=1000 satisfy ?
- `10` -> ge=0 satisfy ?

In [ ]:
from pydantic import ValidationError

print("--- Test 1: name too short ---")
try:
    Item(name="B", price=29.99, quantity=10)  # name length 1 < min_length 2
except ValidationError as e:
    print(e)

print("\n--- Test 2: price = 0 (not > 0) ---")
try:
    Item(name="Book", price=0, quantity=10)   # gt=0 means strictly > 0, 0 not allowed
except ValidationError as e:
    print(e)

print("\n--- Test 3: price > 1000 ---")
try:
    Item(name="Book", price=1500, quantity=10)  # le=1000 means <= 1000, 1500 not allowed
except ValidationError as e:
    print(e)

print("\n--- Test 4: negative quantity ---")
try:
    Item(name="Book", price=29.99, quantity=-1)  # ge=0 means >= 0, -1 not allowed
except ValidationError as e:
    print(e)

### gt vs ge -- Important Difference

```python
quantity: int = Field(gt=0)   # strictly greater than 0  -> 0 NOT allowed, 1 OK
quantity: int = Field(ge=0)   # greater than or equal to 0 -> 0 OK, 1 OK

price: float = Field(lt=1000)  # strictly less than 1000  -> 1000 NOT allowed
price: float = Field(le=1000)  # less than or equal to 1000 -> 1000 OK
```

| Value | `gt=0` | `ge=0` |
|-------|--------|--------|
| -1 | Error | Error |
| 0 | Error | OK |
| 1 | OK | OK |

**Real world usage:**
- Stock quantity: `ge=0` (0 stock ok, negative not ok)
- Discount %: `ge=0, le=100`
- Rating: `ge=1, le=5`
- Price: `gt=0` (price must be positive, 0 not allowed)

## 11) Field() -- description, default, default_factory & model_json_schema()

Previous section lo `Field()` constraints (min_length, gt, ge...) chusam.
Ikkada inkaa 3 important `Field()` features chuddam:

| Feature | Syntax | Meaning |
|---------|--------|--------|
| **Required** | `Field(...)` | `...` (Ellipsis) = must provide, no default |
| **Static default** | `Field(default=18)` | Not provided aithe `18` use chestundi |
| **Dynamic default** | `Field(default_factory=lambda: ...)` | Not provided aithe factory function run chesi value generate chestundi |
| **Description** | `Field(description='...')` | Field ki human-readable explanation -- schema/docs lo use avutundi |

### default vs default_factory

```python
# Static default -- same value anni instances ki
age: int = Field(default=18)
# user1.age == 18, user2.age == 18  (same object shared)

# Dynamic default -- prathi instance ki new value generate
email: str = Field(default_factory=lambda: 'user@example.com')
# Prathi time factory function call avutundi -- mutable defaults ki safe
```

**Analogy:**
> `default=18` ante notice board lo fixed ga `18` raasindi.
> `default_factory` ante receptionist ki cheppinamu -- "oka vellochindhi ki fresh form prepare cheyyi" ani.

In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(..., description="Unique username for the user")          # ... = required
    age: int      = Field(default=18, description="User age, defaults to 18")       # static default
    email: str    = Field(default_factory=lambda: "user@example.com",               # dynamic default
                          description="Default email address")

# user1 -- only required field (username)
user1 = User(username="alice")
print(user1)   # age=18  email='user@example.com'  (both defaults used)

# user2 -- all fields provided
user2 = User(username="bob", age=25, email="bob@domain.com")
print(user2)   # all provided values used

### Output Explanation

```
username='alice' age=18 email='user@example.com'
username='bob'   age=25 email='bob@domain.com'
```

- `user1`: only `username` provided
  - `age` -> `Field(default=18)` -> `18` used
  - `email` -> `Field(default_factory=lambda: 'user@example.com')` -> factory called -> `'user@example.com'` used
- `user2`: anni fields provided -> defaults override ayyay

### Field(...)  -- Ellipsis means Required

```python
username: str = Field(...)   # ... ante required -- provide cheyyakapote ValidationError
username: str = Field()      # same as Field(...) -- also required
username: str                # no Field() -- also required
```

`...` (three dots = Ellipsis) Python built-in object -- Pydantic lo "this field is mandatory" ani signal.

In [ ]:
from pydantic import ValidationError

# Required field miss chesthe -- ValidationError
try:
    User()  # username provide cheyyaledu
except ValidationError as e:
    print("Missing required field:")
    print(e)

## 12) model_json_schema() -- API Documentation Schema

`model_json_schema()` -> Pydantic model nundi **JSON Schema** generate chestundi.

JSON Schema ante oka standard format -- APIs, OpenAPI docs, form validation, frontend validation anni idi use chestay.

**`description` field importance:** `Field(description='...')` lo raasina text JSON schema lo `'description'` key lo kanipistundi. Idi API docs lo automatically appear avutundi (FastAPI tho use chesthe Swagger UI lo kanipistundi).

In [ ]:
import json as json_module

# model_json_schema() -- complete schema generate chestundi
schema = User.model_json_schema()

# Pretty print for readability
print(json_module.dumps(schema, indent=2))

### Schema Output Explanation

```json
{
  "title": "User",
  "type": "object",
  "properties": {
    "username": {
      "title": "Username",
      "description": "Unique username for the user",   <- Field(description=...) lo raasindi
      "type": "string"
    },
    "age": {
      "title": "Age",
      "description": "User age, defaults to 18",
      "default": 18,                                   <- default value schema lo kuda untundi
      "type": "integer"
    },
    "email": {
      "title": "Email",
      "description": "Default email address",
      "type": "string"
    }
  },
  "required": ["username"]                             <- required fields list -- only username
}
```

**Key things in schema:**
- `required`: `["username"]` -- only `username` required, baaki optional (defaults unnay)
- `description` field lo raasina text here visible
- `default: 18` schema lo kuda reflect avutundi
- FastAPI lo idi automatically Swagger UI docs generate cheyyadaniki use avutundi

## Summary

| Concept | What it does |
|---------|-------------|
| `BaseModel` | Parent class -- inherit chesi data model define cheyyadam |
| Type hints (`name: str`) | Pydantic ki field type cheppataniki |
| Auto conversion | `'35'` -> `35` -- reasonable conversions automatic |
| `ValidationError` | Wrong data = immediate clear error |
| `model_dump()` | Object -> Python dict |
| `model_dump_json()` | Object -> JSON string |
| `model_validate()` | Dict -> Object |

**Bottom line:** `BaseModel` inherit chesthe -- type safety, auto conversion, clear errors anni free ga vastay. Manual validation code raayakarledu.